# Exploratory Data Analysis & Preprocessing Strategy
## Cervical Cancer Risk Factors Dataset

**Centrale Casablanca — Coding Week 09-15 March 2026**  
**k. Zerhouni & Team**

---

## Introduction

Avant d'entrainer un modele de machine learning, il est indispensable de comprendre les donnees sur lesquelles il va apprendre. Cette etape, appelee **Exploratory Data Analysis (EDA)**, permet de detecter les problemes inherents au dataset et de choisir des strategies de preprocessing adaptees.

Dans ce notebook, nous analysons le dataset `risk_factors_cervical_cancer.csv`, qui contient des donnees cliniques sur **858 patientes** et **36 features** (facteurs de risque). L'objectif est de predire la variable cible `Biopsy` :
- **0** : No risk (pas de cancer detecte)
- **1** : At risk (cancer detecte)

Nous repondons aux **4 questions critiques** imposees par le sujet :

| # | Probleme detecte | Impact si ignore | Strategie adoptee |
|---|---|---|---|
| 1 | **Missing Values** | Le modele plante ou apprend sur des donnees fausses | Imputation par la mediane |
| 2 | **Outliers** | Les arbres de decision sont biaises par les extremes | Suppression IQR x 1.5 |
| 3 | **Class Imbalance** | Le modele predit toujours la classe majoritaire | Class Weighting |
| 4 | **Correlation** | Redondance entre features, modele moins interpretable | Suppression des features trop correlees (seuil 0.9) |


In [3]:
# ── Imports des bibliotheques ─────────────────────────────────────────────────
# pandas  : manipulation des tableaux de donnees (DataFrame)
# numpy   : operations numeriques sur les tableaux
# sklearn : outils de preprocessing (imputation, split, normalisation)
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# ── Import des fonctions de notre pipeline ────────────────────────────────────
# Ces fonctions sont definies dans src/data_processing.py
# Elles encapsulent toute la logique de nettoyage et de preparation des donnees
import sys
sys.path.append('../src')
from data_processing import (
    load_and_clean_data,      # charge le CSV et convertit les '?' en NaN
    remove_outliers_iqr,      # supprime les lignes avec valeurs aberrantes
    optimize_memory,          # reduit l'empreinte memoire du DataFrame
    supprimer_colonnes_zero   # supprime les colonnes entiererement egales a 0
)

# ── Chargement du dataset brut ────────────────────────────────────────────────
# On charge les donnees dans leur etat original, SANS aucun pretraitement
# Cela nous permet d'observer les vrais problemes du dataset
df_raw = load_and_clean_data('../data/risk_factors_cervical_cancer.csv')

print(f'Dataset charge : {df_raw.shape[0]} patientes, {df_raw.shape[1]} features')
df_raw.head()

Dataset charge : 451 patientes, 30 features


,Age,Number of sexual partners,First sexual intercourse,Num of pregnancies,Smokes,Smokes (years),Smokes (packs/year),Hormonal Contraceptives,Hormonal Contraceptives (years),IUD,...,STDs:Hepatitis B,STDs:HPV,STDs: Time since first diagnosis,Dx:Cancer,Dx:CIN,Dx:HPV,Hinselmann,Schiller,Citology,Biopsy
0,18,4.0,15.0,1.0,0.0,0.0,0.0,0.0,0.00,0.0,...,0.0,0.0,NaN,0,0,0,0,0,0,0
1,15,1.0,14.0,1.0,0.0,0.0,0.0,0.0,0.00,0.0,...,0.0,0.0,NaN,0,0,0,0,0,0,0
2,45,1.0,20.0,5.0,0.0,0.0,0.0,0.0,0.00,0.0,...,0.0,0.0,NaN,1,0,1,0,0,0,0
3,43,2.0,18.0,5.0,0.0,0.0,0.0,0.0,0.00,1.0,...,0.0,0.0,NaN,0,0,0,0,0,0,0
4,41,4.0,21.0,3.0,0.0,0.0,0.0,1.0,0.25,0.0,...,0.0,0.0,NaN,0,0,0,0,0,0,0


---

## 1. Missing Values — Valeurs Manquantes

### Qu'est-ce qu'une valeur manquante ?

Une **valeur manquante** (ou NaN — *Not a Number*) est une cellule du tableau de donnees dont la valeur est absente. Cela peut arriver pour de nombreuses raisons dans un contexte medical :
- La patiente a refuse de repondre a certaines questions (vie privee)
- Le medecin n'a pas realise certains examens
- Une erreur de saisie lors de l'enregistrement des donnees

### Particularite de ce dataset

Dans notre fichier CSV, les valeurs manquantes ne sont **pas representees par des cellules vides**, mais par le caractere **`?`**. Pandas ne les reconnait donc pas automatiquement comme NaN. C'est pourquoi notre fonction `load_and_clean_data()` effectue cette conversion des le chargement :

```python
# Extrait de src/data_processing.py
def load_and_clean_data(filepath):
    df = pd.read_csv(filepath)
    
    # Etape 1 : remplace tous les '?' par NaN (valeur manquante standard de pandas)
    # Etape 2 : convertit toutes les colonnes en type numerique
    #           errors='coerce' transforme en NaN les valeurs non-convertibles
    df = df.replace('?', np.nan).apply(pd.to_numeric, errors='coerce')
    
    return df
```

### Pourquoi les valeurs manquantes sont-elles problematiques ?

La plupart des algorithmes de machine learning ne peuvent pas traiter directement des valeurs NaN. Si on les ignore, le modele plantera a l'entrainement. Il faut donc decider comment les gerer avant d'entrainer.

### Trois options possibles

**Option 1 — Supprimer les lignes avec NaN :**  
Simple mais dangereux. Avec seulement 55 cas positifs (At risk) sur 858, supprimer des lignes risque d'eliminer une grande partie des cas rares que le modele doit apprendre a reconnaitre. Cette option est donc **rejetee**.

**Option 2 — Imputer par la moyenne :**  
On remplace chaque NaN par la valeur moyenne de sa colonne. Probleme : la moyenne est tres sensible aux valeurs extremes (outliers). Dans un contexte medical ou certaines valeurs peuvent etre anormalement elevees, la moyenne est peu representative. Cette option est **rejetee**.

**Option 3 — Imputer par la mediane :**  
On remplace chaque NaN par la valeur mediane de sa colonne. La mediane est le point central de la distribution — elle est **robuste aux outliers** car elle ne tient pas compte des valeurs extremes. C'est l'option **choisie**.

### Notion importante : le Data Leakage

Le **data leakage** (fuite de donnees) est une erreur classique en machine learning. Elle se produit quand des informations du jeu de test influencent l'entrainement, ce qui donne des resultats artificiellement bons qui ne se generaliseront pas en production.

Pour eviter cela, l'imputation doit etre **fittee uniquement sur X_train**, puis appliquee (transform) sur X_test. Voici comment notre pipeline le fait :

```python
# Extrait de src/data_processing.py — fonction preprocess_data()

imputer = SimpleImputer(strategy='median')

# fit_transform sur X_train : calcule la mediane de chaque colonne SUR LES DONNEES D'ENTRAINEMENT
# et remplace immediatement les NaN par ces medianes
X_train_imputed = imputer.fit_transform(X_train)

# transform sur X_test : applique les MEMES medianes calculees sur X_train
# On ne recalcule PAS la mediane sur X_test → pas de data leakage
X_test_imputed = imputer.transform(X_test)
```

Si on faisait `fit_transform` sur X_test aussi, les medianes du test set influenceraient l'imputation, ce qui constituerait un data leakage.

---

## 2. Outliers — Valeurs Aberrantes

### Qu'est-ce qu'un outlier ?

Un **outlier** (ou valeur aberrante) est une observation qui s'ecarte significativement du reste des donnees. Dans un dataset medical, un outlier peut etre :
- Une **erreur de saisie** (ex: une patiente agee de 200 ans)
- Un **cas extremement rare** mais reel (ex: une patiente ayant eu 30 partenaires sexuels)
- Une **anomalie biologique** reelle qui merite d'etre traitee avec precaution

### Pourquoi les outliers sont-ils problematiques ?

Les valeurs extremes peuvent nuire a la qualite du preprocessing et de l'apprentissage :
1. **Biaiser les imputations** : la mediane est robuste, mais des outliers extremes peuvent tout de meme affecter la distribution
2. **Reduire la generalisation** : le modele apprend des regles trop specifiques aux cas extremes
3. **Perturber les correlations** : les outliers peuvent creer de fausses correlations entre features

### Methode choisie : IQR x 1.5 (Interquartile Range)

La methode IQR est la plus utilisee en statistiques pour detecter les outliers de maniere objective. Elle repose sur les **quartiles** de la distribution :

- **Q1** (1er quartile) : 25% des valeurs sont en dessous
- **Q3** (3eme quartile) : 75% des valeurs sont en dessous
- **IQR** = Q3 - Q1 : l'etendue de la zone centrale (50% des donnees)

Toute valeur qui depasse ces bornes est consideree comme un outlier et est **supprimee** :
- Borne inferieure = Q1 - 1.5 x IQR
- Borne superieure = Q3 + 1.5 x IQR

La regle des 1.5 x IQR est une **convention statistique standard** (introduite par John Tukey en 1977). Elle est ni trop stricte (1x IQR supprimerait trop de donnees) ni trop permissive (2x IQR laisserait passer trop d'outliers).

```python
# Extrait de src/data_processing.py
def remove_outliers_iqr(df):
    # On cible uniquement les colonnes continues (valeurs numeriques variees)
    # Les colonnes binaires (0 ou 1) sont EXCLUES : elles ne peuvent pas avoir d'outliers
    cols_a_verifier = [
        'Age',
        'Number of sexual partners',
        'First sexual intercourse',
        'Num of pregnancies',
        'Smokes (years)',
        'Hormonal Contraceptives (years)'
    ]

    for col in cols_a_verifier:
        Q1  = df[col].quantile(0.25)  # 1er quartile
        Q3  = df[col].quantile(0.75)  # 3eme quartile
        IQR = Q3 - Q1                 # etendue interquartile

        lower_bound = Q1 - 1.5 * IQR  # borne inferieure
        upper_bound = Q3 + 1.5 * IQR  # borne superieure

        # On ne conserve que les lignes dont la valeur est DANS les bornes
        df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]

    return df
```

### Pourquoi exclure les colonnes binaires ?

Les colonnes binaires comme `STDs`, `Schiller`, `Hinselmann`, `Citology` ne peuvent prendre que les valeurs **0 ou 1**. Il est donc mathematiquement impossible d'avoir un outlier sur ces colonnes — Q1, Q3 et IQR ne sont pas des mesures pertinentes pour des distributions binaires.

### Alternatives considerees

**Winsorisation :** Au lieu de supprimer les outliers, on les **remplace par la valeur de la borne** (on les "ecrete"). Cette methode conserve toutes les lignes mais modifie les valeurs reelles. Dans un contexte medical, modifier les valeurs cliniques d'une patiente (meme extremes) peut introduire un biais, donc cette option est **rejetee**.

**Conservation totale :** Ne rien faire. Si certaines valeurs extremes sont de vraies erreurs de saisie (ex: age = 200), les conserver degradera la qualite du modele. Option **rejetee**.

---

## 3. Class Imbalance — Desequilibre des Classes

### Qu'est-ce que le desequilibre des classes ?

Le **desequilibre des classes** (*class imbalance*) se produit quand une classe de la variable cible est beaucoup plus frequente que l'autre. Dans notre dataset :
- **Classe 0 (No risk)** : 803 patientes — **93.6%** du dataset
- **Classe 1 (At risk)** : 55 patientes — **6.4%** du dataset

Le ratio est de **14.6 : 1** — pour chaque patiente malade, il y en a 14 en bonne sante.

### Pourquoi le desequilibre est-il dangereux ?

Imaginons un modele naif qui predit toujours **"No risk"** pour toutes les patientes. Ce modele aurait une **accuracy de 93.6%**, ce qui semble excellent — mais il ne detecterait **aucun cancer**. C'est exactement ce comportement que l'on veut eviter.

Sans traitement, un modele va naturellement "apprendre" a privilegier la classe majoritaire car cela minimise son erreur globale. Le modele va donc :
- Bien predire les cas "No risk" (classe majoritaire)
- **Rater la plupart des cas "At risk"** (classe minoritaire)

### La metrique a privilegier : le Rappel (Recall)

En contexte medical, **toutes les erreurs ne se valent pas** :
- **Faux negatif** (predire "No risk" alors que la patiente est malade) : la patiente ne recoit pas de traitement → **consequence grave, potentiellement fatale**
- **Faux positif** (predire "At risk" alors que la patiente est saine) : la patiente fait des examens supplementaires inutiles → **consequence benigne**

On utilise donc le **Rappel** (ou sensibilite) comme metrique principale, plutot que l'accuracy :

> **Rappel = Vrais Positifs / (Vrais Positifs + Faux Negatifs)**

### Strategies disponibles

**Option 1 — Undersampling :**  
On reduit la classe majoritaire en supprimant des exemples "No risk" jusqu'a atteindre un equilibre. Probleme : on **jette de l'information** precieuse. Avec seulement 858 lignes au depart, on ne peut pas se permettre d'en supprimer des centaines. Option **rejetee**.

**Option 2 — Class Weighting :**  
On attribue un **poids plus eleve** a la classe minoritaire lors de l'entrainement, sans modifier les donnees. Le modele penalise davantage les erreurs sur les cas 'At risk'. Probleme : pour un desequilibre aussi extreme (14.6:1), les poids seuls peuvent etre insuffisants pour forcer le modele a apprendre correctement la classe minoritaire. Option **rejetee**.

**Option 3 — Oversampling manuel (choisi) :**  
On **duplique aleatoirement** des exemples existants de la classe minoritaire jusqu'a atteindre le meme nombre que la classe majoritaire. On ne cree pas de nouvelles donnees fictives — on reutilise des donnees reelles. C'est l'option **choisie**.

```python
# Extrait de src/data_processing.py — fonction preprocess_data()

# --- DEBUT OVERSAMPLING MANUEL ---
# On separe les classes du set d'entrainement
X_train_pos = X_train_imp[y_train == 1]   # exemples positifs (At risk)
X_train_neg = X_train_imp[y_train == 0]   # exemples negatifs (No risk)

# On calcule la taille de la classe majoritaire
num_neg = len(X_train_neg)

# np.random.choice : tire aleatoirement des indices parmi les exemples positifs
# size=num_neg  : on tire autant d'exemples que la classe negative
# replace=True  : on peut tirer le meme exemple plusieurs fois (duplication)
np.random.seed(42)  # graine fixe pour la reproductibilite
indices = np.random.choice(len(X_train_pos), size=num_neg, replace=True)
X_train_pos_over = X_train_pos[indices]

# np.vstack : empile verticalement les deux tableaux (concatenation de lignes)
# np.hstack : concatene horizontalement les deux vecteurs de labels
X_train_bal = np.vstack((X_train_neg, X_train_pos_over))         # 50% No risk, 50% At risk
y_train_bal = np.hstack((np.zeros(num_neg), np.ones(num_neg)))   # labels correspondants
# --- FIN OVERSAMPLING ---
```

### Pourquoi appliquer l'oversampling UNIQUEMENT sur X_train ?

C'est une regle fondamentale : **X_test ne doit jamais etre modifie**. Le test set doit refleter la distribution reelle du probleme pour evaluer honnetement les performances du modele. Si on equilibrait aussi le test set, on obtiendrait des metriques trop optimistes qui ne se reproduiraient pas en production.


### Conclusion — Class Imbalance

| Strategie | Decision | Raison |
|---|---|---|
| Sans traitement | Rejete | Le modele ignore les cas positifs |
| Undersampling | Rejete | Perte massive d'information |
| SMOTE | Rejete | Exemples synthetiques, bruit medical |
| **Oversampling manuel** | **Choisi** | Donnees reelles, equilibre 1:1, pas de data leakage |

| Critere | Valeur |
|---|---|
| Desequilibre initial | 14.6 : 1 |
| Strategie choisie | Duplication aleatoire avec `replace=True` sur X_train |
| Equilibre obtenu | 1 : 1 (50% No risk, 50% At risk) |
| Graine fixe | `np.random.seed(42)` pour la reproductibilite |


---

## 4. Correlation — Redondance entre Features

### Qu'est-ce que la correlation ?

La **correlation** mesure le degre de relation lineaire entre deux variables numeriques. Le **coefficient de Pearson** varie entre -1 et +1 :
- **+1** : correlation positive parfaite (quand l'une augmente, l'autre augmente proportionnellement)
- **-1** : correlation negative parfaite (quand l'une augmente, l'autre diminue proportionnellement)
- **0** : aucune relation lineaire

Dans notre dataset, on peut s'attendre a des correlations entre des features comme `STDs (number)` et `STDs:condylomatosis` — car avoir plusieurs MST est naturellement correle avec chaque MST specifique.

### Pourquoi la correlation est-elle problematique ?

Des features fortement correlees apportent **une information redondante** au modele. Les problemes sont :
1. **Instabilite des coefficients** : si deux features donnent la meme information, le modele peut leur attribuer des importances erratiques d'un entrainement a l'autre
2. **Overfitting** : le modele apprend des combinaisons specifiques de features correlees qui ne se generalisent pas
3. **Interpretabilite reduite** : il devient difficile de savoir quelle feature est vraiment importante

### Premiere etape : supprimer les colonnes sans variance

Avant d'analyser les correlations, il faut d'abord supprimer les colonnes dont **toutes les valeurs sont egales a 0**. Ces colonnes n'apportent aucune information au modele (variance nulle) et peuvent meme perturber le calcul des correlations.

```python
# Extrait de src/data_processing.py
def supprimer_colonnes_zero(df):
    # Identification des colonnes ou TOUTES les valeurs sont egales a 0
    colonnes_a_supprimer = [col for col in df.columns if (df[col] == 0).all()]
    df_nettoye = df.drop(columns=colonnes_a_supprimer)
    return df_nettoye
```

### Deuxieme etape : supprimer les features trop correlees

Une fois les colonnes nulles supprimees, on applique une **suppression automatique des features hautement correlees** via la fonction `drop_high_correlation()` :

```python
def drop_high_correlation(df, threshold=0.9):
    # Calcul de la matrice de correlation
    corr_matrix = df.corr().abs()
    
    # Selection de la partie superieure de la matrice
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    
    # Identification des colonnes a supprimer
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    
    if to_drop:
        print(f'Colonnes supprimees car trop correlees : {to_drop}')
        return df.drop(columns=to_drop)
    return df
```

### Comment fonctionne cette methode ?

La logique repose sur une propriete fondamentale de la matrice de correlation : elle est **symetrique** — la correlation de A avec B est exactement la meme que celle de B avec A. Pour eviter de traiter chaque paire deux fois, on ne regarde que le **triangle superieur** de la matrice (les cellules au-dessus de la diagonale, `k=1` exclut la diagonale elle-meme).

On prend les valeurs absolues (`abs()`) car une correlation de -0.95 est aussi problematique qu'une correlation de +0.95 — dans les deux cas, les deux features sont quasi-redondantes.

Pour chaque colonne du triangle superieur, si **au moins une** de ses correlations avec une autre feature depasse le seuil de 0.9, la colonne est marquee pour suppression. Ce seuil de 0.9 est deliberement conservateur : on ne supprime que les redondances quasi-parfaites, pas les correlations moderees qui peuvent apporter des perspectives complementaires.

### Pourquoi ce choix est-il justifie ?

Une correlation superieure a 0.9 signifie que deux features partagent plus de **81% de leur variance** (`r² > 0.81`). Dans ce cas, garder les deux n'apporte pratiquement aucune information supplementaire au modele — on ne fait qu'augmenter le bruit et la complexite du preprocessing. La suppression est donc la reponse la plus directe et la plus propre.

Par rapport a d'autres approches :

**PCA rejetee :** La PCA transforme les features correlees en composantes principales independantes. Si elle resout bien le probleme mathematiquement, elle detruit l'**interpretabilite medicale** — on ne peut plus relier les features aux facteurs cliniques reels.

**Conservation totale rejetee :** Ne rien faire laisse des features quasi-identiques dans le dataset, ce qui peut rendre le modele instable et difficile a interpreter.

**Suppression manuelle rejetee :** Choisir manuellement quelles features supprimer introduit un biais humain. La methode `drop_high_correlation()` est **automatique, reproductible et objective** — elle applique le meme critere numerique a toutes les paires de features sans jugement subjectif.


### Conclusion — Correlation

| Decision | Justification |
|---|---|
| **`supprimer_colonnes_zero()`** | Elimine les features sans variance, inutiles et perturbent les correlations |
| **`drop_high_correlation(threshold=0.9)`** | Supprime automatiquement les features quasi-redondantes (r² > 0.81) |
| PCA rejetee | Perte totale de l'interpretabilite medicale des features |
| Conservation totale rejetee | Features redondantes → instabilite et bruit inutile |
| Suppression manuelle rejetee | Subjective et non reproductible |

---

## Resume General — Decisions de Preprocessing

| Probleme | Detecte ? | Fonction | Strategie | Resultat |
|---|---|---|---|---|
| **Valeurs manquantes** | Oui — 26/36 colonnes | `load_and_clean_data()` + `SimpleImputer(median)` | Imputation par la mediane sur X_train uniquement | 0 NaN a l'entrainement, pas de data leakage |
| **Outliers** | Oui — sur 6 colonnes continues | `remove_outliers_iqr()` | Suppression IQR x 1.5, colonnes binaires exclues | Dataset nettoye, distribution preservee |
| **Desequilibre classes** | Oui — ratio 14.6:1 | `preprocess_data()` | Oversampling manuel sur X_train uniquement | Equilibre 1:1, donnees reelles, pas de data leakage |
| **Correlations** | Oui — features STDs liees | `supprimer_colonnes_zero()` + `drop_high_correlation()` | Suppression colonnes nulles + suppression features r > 0.9 | Redondance eliminee, interpretabilite preservee |
